# Describing a single variable

*Distributions, summary statistics, and outliers*

With this chapter we begin Part II, the descriptive stage of the course: describing what happened. Our starting point is the simplest unit of description, a single variable. For Prairie Wholesale, this starting point becomes a concrete question: what does a typical order look like? Answering the question requires a picture of the distribution, a choice between two competing definitions of "typical", and a policy for one very large order placed by a customer. Along the way we introduce Plotly, the charting library we use for every figure in the rest of this book.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-04-single-variable.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-04-single-variable.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly
import sys
if "google.colab" in sys.modules:
    !git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

> **Tools in this chapter**
>
> | Tool | Why we use it here | Alternatives | Trade-off |
> |---|---|---|---|
> | Plotly (`plotly.express`) | Interactive charts with one function call per chart type; exact values appear on hover | matplotlib (static, ubiquitous), Altair | Plotly's interactivity fits notebooks and dashboards; much legacy code is written in matplotlib |
>
> : {tbl-colwidths="[12,33,22,33]"}

## From lines to orders {#sec-ch4-intro}

We have two imports in this chapter, one of which is new. **Plotly** is a free to use open-source charting library, hosted by Plotly Inc. We chart with its `plotly.express` module (imported as `px` by convention). It provides one function call per chart type and interactive output by default, so that in the online edition hovering over a figure shows the underlying values. 

In [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) we worked with the order-line data. "What does a typical order look like?" is a question about complete orders, so we begin with an aggregation: one row per order, with its total.

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])

order_totals = orders.groupby("order_id").agg(
    total=("line_total", "sum"),
    date=("order_date", "first"),
    channel=("channel", "first"),
    customer_id=("customer_id", "first"),
)
order_totals["total"].describe().round(2)

Before creating any chart, we note two numbers from this summary. The mean order is \$458, and the median order is about \$168. Both are valid answers to "what is a typical order?", yet they differ by a factor of almost three. A gap this large indicates that order sizes are highly uneven: a small number of very large orders pulls the mean far above the middle of the distribution. In the remainder of this chapter, we locate those large orders and decide how to report summaries in their presence.

## Histograms

`describe` summarizes a column in a few numbers. A **histogram** shows the complete distribution. The value range is divided into intervals, called bins, and each bar's height is the number of orders that fall in that bin. We draw one with a single line of Plotly:

In [ ]:
px.histogram(order_totals, x="total", nbins=60,
             labels={"total": "Order total ($)"})

A first histogram of transaction data commonly shows this shape: one big spike and a long, apparently empty tail, since such data usually contains many small transactions and a few very large ones. However, the tail is not empty; it has a few very large orders. Those orders determine the horizontal axis range, so all the small orders fall in the first few bins, and the few large orders form a long tail across the remaining bins. Such a distribution is called **right-skewed**.

To see the shape of the bulk of the orders, we restrict the axis to orders under \$2,000:

In [ ]:
under_2k = order_totals[order_totals["total"] < 2000]

fig = px.histogram(under_2k, x="total", nbins=80,
                   labels={"total": "Order total ($)"})
fig.add_vline(x=order_totals["total"].median(), line_color="#cf222e",
              annotation_text="median", annotation_position="top")
fig.add_vline(x=order_totals["total"].mean(), line_color="#0969da",
              annotation_text="mean", annotation_position="top right")
fig.show()

Reading a histogram requires judgment about the bin width as much as about the shape. Thin bins show sampling noise, while wide bins smooth away real features of the distribution. There is no one true width for all datasets. We try a few widths and report the features that persist at every width.

::: {.content-visible when-format="html"}
The explorer below contains 600 real order totals. Move the slider to add and remove detail. The mean and median markers do not move, because neither statistic depends on the bin width. Only the picture changes.

<iframe src="../../assets/demos/histogram-bins.html" width="100%" height="470" style="border:1px solid #d0d7de; border-radius:8px;" title="Histogram bin width explorer"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an interactive explorer here: a bin-width slider over 600 real order totals, with fixed mean and median markers. @fig-hist-zoom shows one setting.
:::

## Mean versus median

Why do the two "typical" numbers conflict? The **mean** is the sum of every dollar divided by the number of orders, so every dollar has an impact on it. The **median** is the middle order when all orders are sorted, so the median depends on position alone. In a right-skewed distribution the large values in the tail increase the mean. The median does not move much, since half the orders are still smaller than it.

The largest values in the tail are individual orders that we can inspect. We sort them:

In [ ]:
order_totals.sort_values("total", ascending=False).head(3).round(2)

The largest order is about \$66,000, more than double the second largest. In August 2025, one school district placed a district-wide stock-up order: floor cleaner, paper, and supplies for a year, all at once. This single order changes the company-wide mean measurably:

In [ ]:
with_out = order_totals["total"]
without = with_out.drop(with_out.idxmax())   # drop exactly the one largest order

pd.DataFrame({
    "mean": [with_out.mean(), without.mean()],
    "median": [with_out.median(), without.median()],
}, index=["all orders", "without the school order"]).round(2)

One order moves the company-wide mean by about \$1.63. It moves the median by one cent. Statisticians call the median **robust**: resistant to extreme values.

The mean is sensitive to extreme values. Whether this sensitivity helps or misleads depends on how the number will be used. When the quantity of interest is a sum, such as total revenue, the sensitivity is appropriate: the large orders contribute their dollars to the total, and a summary that reflects them is accurate. When the mean stands in for a typical order, the same sensitivity is misleading: a few atypical orders can shift the number toward order sizes that rarely occur, and a decision optimized around such a number is optimized for orders the company rarely receives.

We therefore report the mean for questions about totals, for example "what revenue does an average order bring?", and we report the median for questions about the typical case, for example "what size order should our packing station be designed around?"

::: {.content-visible when-format="html"}
The explorer below shows the difference. Drag the amber point to the right. The mean moves with it across the axis. The median moves very little.

<iframe src="../../assets/demos/mean-median-outlier.html" width="100%" height="290" style="border:1px solid #d0d7de; border-radius:8px;" title="Mean versus median outlier explorer"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an interactive explorer here: fifteen order totals on a number line, one draggable. Dragging it rightward moves the mean far from the other points while the median stays almost fixed.
:::

## Measuring spread

A central value alone does not characterize a distribution. A measure of spread is also required. The two summaries of spread parallel the mean/median pair. The **standard deviation** is based on squared distances to the mean, and thus is sensitive to the tail in the same manner as the mean. The **interquartile range (IQR)** is the distance between the 25th and 75th percentiles, and it is resistant to the tail in the same manner as the median.

In [ ]:
t = order_totals["total"]
q1, q3 = t.quantile(0.25), t.quantile(0.75)
pd.Series({
    "std": t.std(),
    "IQR": q3 - q1,
    "middle half of orders": f"${q1:.0f} to ${q3:.0f}",
})

The standard deviation is over \$1,000 for a median order of \$168. This number mostly restates the skew. The IQR is directly useful to the business: the middle half of Prairie Wholesale's orders are between roughly \$68 and \$432.

The **box plot** shows these robust summaries: the box is the IQR, the line in the middle is the median, and points outside the whiskers are optionally marked as potential outliers. The box plot is most useful for comparing groups. We draw one box per sales channel in our data:

In [ ]:
px.box(order_totals, x="channel", y="total", log_y=True,
       labels={"total": "Order total ($, log scale)", "channel": "Channel"})

We use a log scale here for one reason: on a linear axis, skewed money data is unreadable. A log axis spaces \$10, \$100, and \$1,000 evenly, so the whole distribution becomes visible at once. We use a log scale again in [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html).

::: {.content-visible when-format="html"}
The toggle below shows the same 840 orders on both axes. On the linear axis the channels look similar. Toggle to the log axis and compare again.

<iframe src="../../assets/demos/log-linear.html" width="100%" height="470" style="border:1px solid #d0d7de; border-radius:8px;" title="log linear"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has a toggle here between linear and log axes over 840 real orders; the channel differences that the linear axis compresses become apparent on the log axis.
:::


## An outlier policy

An **outlier** is an observation that lies far from the bulk of the data. "Far" can be defined in a few different ways. The points in @fig-box-channel were marked by a common outlier definition: a point counts as a potential outlier when it falls more than 1.5 IQRs beyond either edge of the box. This convention is called the **1.5 × IQR rule**. The rule is used to mark candidates only; it is your job as the analyst to make the final decision about whether a marked value is an outlier or not. For every candidate outlier, we ask one question: is this an error, or is it valid for our analysis?

Some marked values are errors. A negative quantity or a \$0.001 price is impossible, and we correct or remove such values as part of data cleaning. The school-district order is not an error: the transaction occurred and the revenue was collected, so a summary of the year that excluded this order would misrepresent the year. The professional practice is to keep the order and to match the reported summary to the question: the median and IQR when the question concerns the typical order, and the mean when the question concerns totals. When a single observation moves a headline number, we disclose that fact in a footnote. We never silently drop rows because they are inconvenient; every omission is stated and justified. We return to this large order in [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), where it distorts a regression line.

## Categorical variables: counts and shares

Order totals are numeric. Channel, region, and business type are **categorical**: each value is a label from a fixed set of options, not a quantity. A categorical variable has no mean, median, or histogram. We describe such a variable by counting how many orders fall in each category. `value_counts` returns counts; with `normalize=True` it returns shares:

In [ ]:
customers = pd.read_csv(DATA_DIR / "pw_customers.csv")
customers["business_type"].value_counts(normalize=True).round(3)

In [ ]:
counts = customers["business_type"].value_counts().reset_index()

px.bar(counts, x="count", y="business_type", orientation="h",
       labels={"count": "Customers", "business_type": ""})

One convention applies to all categorical charts: sort the bars by value, unless the categories have a natural order. Note also that a bar chart of counts shows how many customers fall in each group, which is a different question from how much revenue comes from each group. Restaurants are the largest group of accounts. Whether they generate the most revenue is a two-variable question. We take up two-variable questions in [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html).

## Evaluation: which summary is stable enough for the packing station

Having constructed several summaries in this chapter, we now assess whether one of them is fit for a specific use. The warehouse team builds the packing station around a "typical order" number, and a figure that swings whenever a large order arrives would be a poor basis for the design. The team revisits the number periodically, so we test stability directly: we recompute each candidate for every month in the data and measure how much it moves.

|  |  |
|---|---|
| **Metric** | how much the candidate number moves when it is recomputed over normal operation. |
| **Test** | recompute each candidate for every month in the data and measure the gap between its highest and lowest monthly value. |
| **Baseline** | the naïve choice, the mean. |

: {tbl-colwidths="[18,82]"}

In [ ]:
monthly = (order_totals.groupby(order_totals["date"].dt.to_period("M"))["total"]
           .agg(["mean", "median"]))
monthly.index = monthly.index.to_timestamp()

px.line(monthly, labels={"value": "Order total ($)", "date": "Month", "variable": "Summary"})

The gap between the highest and the lowest monthly value, for each candidate:

In [ ]:
(monthly.max() - monthly.min()).round(2)

By the metric we set, the median is the right choice for the packing station: across three years its monthly value stays within a band of about \$50, while the mean moves across a range of \$237, nearly five times as wide. The chart also connects this evaluation to the outlier policy: the mean's largest spike is August 2025, the month of the school-district order. We apply this lesson throughout the book: we judge every summary against the purpose it will be used for, with an evaluation metric chosen to match that purpose.

## The decision this informs

Two decisions follow from one variable. The packing station is designed around the median order, near \$168, and must comfortably handle the middle half of orders, roughly \$68 to \$432 (the IQR). The mean, near \$458, sits above the 75th percentile of order totals: a station designed around the mean would be oversized for roughly three of every four orders. The second decision concerns reporting: monthly revenue reviews present the median alongside the mean, so that a single unusually large order is not mistaken for a company-wide trend.

## Exercises



### Build lab

Describe a different variable end to end: the **discount rate** (`discount_pct`) at the order level. Aggregate to one row per order (the discount is constant within an order, so `first` is a valid aggregation). Produce its `describe`, a histogram with a sensible bin width, and a box plot split by channel. State in two sentences: what shape does the distribution have, and which channel has the heaviest discounts?

### Evaluate lab

In the build lab you chose one bin width for the histogram. Recompute the histogram at half and at double that width. State which features of the shape are present at all three settings and which are present at only some. Then report the mean and median discount, and say which you would put in a pricing review and why, in one sentence each.